# 🌱 Using CNW-Grass with different climate scenarios and parameters

This tutorial will illustrate how to use CNW-Grass for different climate scenarios or parameter values. To that purpose, we will define a list of scenarios, then a generic function that will read the information of a given scenario and which will finally passed these variables to the model by calling a script *main.py*.

First, we will load a csv file defining the list of scenarios to be run. Each scenario has an id and a list of variables contained in the columns to be passed to CNW-Grass.
NB: the last two columns illustrate how model parameters can be modified for a given scenario (for sensitivity analyses for instance).

In [ ]:
import pandas as pd
import os


scenarios = pd.read_csv(os.path.join('inputs', 'scenarios_list.csv'), index_col='Scenario')
scenarios.head()


Let's select a scenario among the 4 proposed. Scenarios differ by soil N content (N%U), the incident PAR (PAR%) and plant density (D%).

*Note that these scenarios are run with the option 'constant_Conc_Nitrates, which allows to simulate hydroponic-like conditions.
You can also see that parameters values can be dynamically passed from the scenario file to the specified submodel ({model_name:paramter_name}).*

In [ ]:
import ipywidgets as widgets
from IPython.display import display


scenario_options = (scenarios.index.astype(str) +
                        ' - ' +
                        scenarios['Scenario_label']).tolist()
selected_scenario_id = widgets.IntText(value=1)

# widget Dropdown
scenario_selector = widgets.Dropdown(
    options=scenario_options,
    value=scenario_options[0], # default scenario
    description='Scenario :',
    disabled=False,
)

def update_scenario(change):
    selected_id = int(change.new.split(' - ')[0])
    selected_scenario_id.value = selected_id

scenario_selector.observe(update_scenario, names='value')

print("Select a scenario :")
display(scenario_selector)


Then, we will define a function *run_cnwgrass* that will load the data from a specific scenario.

In [ ]:
import tools
from openalea.cnwgrass.integration.runner import run as runner


def run_scenario(scenario_data, inputs_dir_path, outputs_dir_path='outputs'):
    """
    Run the main.py of CNW-Grass using data from a specific scenario

    :param pandas.DataFrame scenario_data: the dataframe of the scenario
    :param str inputs_dir_path: the path directory of inputs
    :param str outputs_dir_path: the path to save outputs
    """

    # Path of the directory which contains the inputs of the model
    if inputs_dir_path:
        INPUTS_DIRPATH = inputs_dir_path
    else:
        INPUTS_DIRPATH = scenario_data.get('INPUTS_DIRPATH', 'inputs')


    # -- SIMULATION PARAMETERS --
    scenario_name = str(scenario_data.Scenario_label)

    # Create dict of parameters for the scenario
    scenario_parameters = tools.buildDic(scenario_data.to_dict())

    # Do run the simulation?
    RUN_SIMU = scenario_parameters.get('Run_Simulation', True)

    SIMULATION_LENGTH = scenario_parameters.get('Simulation_Length', 3000)

    # Do run the simulation from the output files ?
    RUN_FROM_OUTPUTS = scenario_parameters.get('Run_From_Outputs', False)

    # Do run the postprocessing?
    RUN_POSTPROCESSING = scenario_parameters.get('Run_Postprocessing', True)

    # Do generate the graphs?
    GENERATE_GRAPHS = scenario_parameters.get('Generate_Graphs', False)

    if RUN_SIMU or RUN_POSTPROCESSING or GENERATE_GRAPHS:

        # -- SIMULATION DIRECTORIES --

        # Path of the directory which contains the outputs of the model
        if not outputs_dir_path:
            OUTPUTS_DIRPATH = 'outputs'
        else:
            OUTPUTS_DIRPATH = outputs_dir_path
        scenario_dirpath = os.path.join(OUTPUTS_DIRPATH, scenario_name)

        if not os.path.exists(OUTPUTS_DIRPATH):
            os.mkdir(OUTPUTS_DIRPATH)

        # Create the directory of the Scenario where results will be stored
        if not os.path.exists(scenario_dirpath):
            os.mkdir(scenario_dirpath)

        # Create directory paths for graphs, outputs and postprocessing of this scenario
        scenario_graphs_dirpath = os.path.join(scenario_dirpath, 'graphs')
        if not os.path.exists(scenario_graphs_dirpath):
            os.mkdir(scenario_graphs_dirpath)
        # Outputs
        scenario_outputs_dirpath = os.path.join(scenario_dirpath, 'outputs')
        if not os.path.exists(scenario_outputs_dirpath):
            os.mkdir(scenario_outputs_dirpath)
        # Postprocessing
        scenario_postprocessing_dirpath = os.path.join(scenario_dirpath, 'postprocessing')
        if not os.path.exists(scenario_postprocessing_dirpath):
            os.mkdir(scenario_postprocessing_dirpath)

        runner(simulation_length=SIMULATION_LENGTH, forced_start_time=0,
                  run_simu=RUN_SIMU, run_postprocessing=RUN_POSTPROCESSING,generate_graphs=GENERATE_GRAPHS, run_from_outputs=RUN_FROM_OUTPUTS,
                  show_3Dplant=False, heterogeneous_canopy=True,
                  INPUTS_DIRPATH=INPUTS_DIRPATH,
                  METEO_FILENAME=scenario_data.get('METEO_FILENAME'), MANAGEMENT_FILENAME=scenario_data.get('Management_filename'),
                  GRAPHS_DIRPATH=scenario_graphs_dirpath,
                  OUTPUTS_DIRPATH=scenario_outputs_dirpath,
                  POSTPROCESSING_DIRPATH=scenario_postprocessing_dirpath,
                  update_parameters_all_models=scenario_parameters)
    return scenario_name


Finally, let's run a scenario from the list (wait for the end of simulation).

In [ ]:
scenario_id = selected_scenario_id.value
scenario_name = run_scenario(inputs_dir_path='inputs', outputs_dir_path='outputs', scenario_data=scenarios.loc[scenario_id])


Let's visualise some graphs

In [ ]:
import matplotlib.pyplot as plt


axes_postprocessing_df = pd.read_csv(os.path.join('outputs', scenario_name, 'postprocessing', 'axes_postprocessing.csv'))
organs_postprocessing_df = pd.read_csv(os.path.join('outputs', scenario_name, 'postprocessing', 'organs_postprocessing.csv'))
roots_postprocessing_df = organs_postprocessing_df[organs_postprocessing_df['organ'] == 'roots']
hz_postprocessing_df = pd.read_csv(os.path.join('outputs', scenario_name, 'postprocessing', 'hiddenzones_postprocessing.csv'))

# Biomass allocation
plt.figure(figsize=(10, 6))
plt.plot(axes_postprocessing_df['t'], axes_postprocessing_df['sum_dry_mass_roots'], label='Root biomass')
plt.plot(axes_postprocessing_df['t'], axes_postprocessing_df['sum_dry_mass_shoot'], label='Shoot biomass')
plt.xlabel('Time (hour)')
plt.ylabel('Dry mass (g)')
plt.title('Biomass allocation')
plt.legend()
plt.grid(True)
plt.show()

# C and N acquisition
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(axes_postprocessing_df['t'], axes_postprocessing_df['Total_Photosynthesis'], label='Photosynthesis', color='yellow')
ax1.set_xlabel('Time (hour)')
ax1.set_ylabel('Photosynthesis (µmol C)')

ax2 = ax1.twinx()
ax2.plot(roots_postprocessing_df['t'], roots_postprocessing_df['Uptake_Nitrates'], label='N uptake', color='green')
ax2.set_ylabel('N Uptake (µmol N)')

plt.title('C and N acquisition')
fig.legend(loc='upper right')
plt.grid(True)
plt.show()

# Leaf length
plt.figure(figsize=(10, 6))
for metamer in hz_postprocessing_df['metamer'].unique():
    leaf = hz_postprocessing_df[(hz_postprocessing_df['axis'] == 'MS') & (hz_postprocessing_df['metamer'] == metamer)]
    plt.plot(leaf['t'], leaf['leaf_L'], label=metamer)

plt.xlabel('Time (hour)')
plt.ylabel('Leaf length (m)')
plt.legend(loc='center right', title='Leaf number')
plt.grid(True)
plt.show()
